# 04 — F-I Curve

Firing rate vs injected current. Classic HH exhibits class-2 excitability
(discontinuous firing onset). Deterministic and stochastic curves.

In [1]:
import sys; sys.path.insert(0, '/workspace')
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIG_DIR = '/mnt/results/hh_simulator/figures'
os.makedirs(FIG_DIR, exist_ok=True)
from hh_simulator import (NaChannel, KChannel, LeakChannel, PointCell,
    Simulator, analysis, viz)
cell = PointCell([NaChannel('classic'), KChannel('classic'), LeakChannel()])
sim = Simulator(cell)
I_range = np.arange(0, 25, 1.0)

In [2]:
I_det, rate_det = analysis.fi_curve(sim, I_range, t_span=(0,300),
    onset=20, steady_window=(100,300))
viz.plot_fi_curve(I_det, rate_det, savepath=f'{FIG_DIR}/04_fi_deterministic.svg')
plt.show()

In [3]:
I_stoch = np.arange(0, 25, 2.0)
trials = []
for seed in range(4):
    rng = np.random.default_rng(seed)
    I_s, rate_s = analysis.fi_curve(sim, I_stoch, t_span=(0,300),
        onset=20, steady_window=(100,300), mode='stochastic',
        dt=0.01, N_channels={'Na':5000,'K':1500}, rng=rng)
    trials.append(rate_s)
trials = np.array(trials)
fig, ax = plt.subplots(figsize=(5,3.5))
ax.errorbar(I_stoch, trials.mean(0), yerr=trials.std(0), fmt='o-',
            color='#FF9400', lw=1.5, capsize=3, label='stochastic (N=5000)')
ax.plot(I_det, rate_det, 'k-', lw=1, alpha=0.5, label='deterministic')
ax.set_xlabel('Injected current (uA/cm^2)'); ax.set_ylabel('Firing rate (Hz)')
ax.set_title('F-I curve: deterministic vs stochastic'); ax.legend(frameon=False)
fig.savefig(f'{FIG_DIR}/04_fi_comparison.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/04_fi_comparison.png', bbox_inches='tight', dpi=150)
plt.show()